# Rarity-driven sparse retrieval — diagnostics

| Section | What runs | Gate |
|---|---|---|
| Step 1 | `rarity` — IDF over the pool, irregular-term list | terms are scriptural vocabulary, not tokenizer debris |
| Step 3 | `leakage` — near-duplicate audit, quarantine | flag count small enough that quarantining leaves the pool intact |
| Step 2 | `sparse_select` — greedy coverage dry run | the channel fires often enough, and is less redundant than dense |


In [8]:
# e5-large in fp32 over a 10.8k-row pool; any Colab GPU is enough.
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA GeForce RTX 4060 Laptop GPU, 8188 MiB


In [9]:
%cd /home/prnamhr/projects/Style-Aware-MT
!pip install -r requirements.txt

# Text-only pipeline; these two carry an ABI mismatch against the pinned torch.
!pip uninstall -y torchvision torchaudio

/home/prnamhr/projects/Style-Aware-MT

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


`data/knn_index/` is git-ignored, so the pool index is rebuilt each session. The register
centroid is committed and already present.

In [10]:
!python3 manage.py build_index --config configs/base_qwen.yaml

Embedding 10860 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Batches: 100%|███████████████████████████████| 340/340 [00:13<00:00, 25.02it/s]
Wrote index to data/knn_index/ : embeddings (10860, 1024), 10860 pairs


---
## Step 1 — the rarity list

IDF over the source side of `data/splits/train.jsonl`, keeping the rarest terms.

In [11]:
!python3 manage.py rarity --config configs/sparse_retrieval.yaml

Computing IDF over 10860 pool sources (zwnj=keep) ...
  22789 pool terms, 22789 at df>=1 -> 11668
  requested top 20%, realized 51.2% (df <= 1)
  NOTE: the rank cut fell inside an IDF tie block and extended to its edge.
        Raise --min_df to move the cut off the hapax block.
  hapax share 100.0%
  df histogram (irregular): {'1': 11668, '2': 0, '3': 0, '4-9': 0, '10-99': 0, '100+': 0}
  ZWNJ variant collisions: 25 [('آسود\u200cگی', 'آسودگی'), ('الذین\u200cهم', 'الذینهم'), ('انشآء\u200cالله', 'انشآءالله')]
Wrote results/rarity_train.json and results/rarity_train_sample50.tsv


In [12]:
import json

import pandas as pd

rarity = json.load(open('results/rarity_train.json'))
print(f"{rarity['n_terms']} pool terms -> {rarity['n_irregular']} irregular "
      f"(requested {rarity['config']['top_frac']:.0%}, realized {rarity['realized_frac']:.1%}, "
      f"df <= {rarity['cutoff_df']})")
print('pool df histogram     :', rarity['df_histogram']['pool'])
print('irregular df histogram:', rarity['df_histogram']['irregular'])

sample = pd.read_csv('results/rarity_train_sample50.tsv', sep='\t')
sample['example'] = sample['example'].str.slice(0, 60)
sample

22789 pool terms -> 11668 irregular (requested 20%, realized 51.2%, df <= 1)
pool df histogram     : {'1': 11668, '2': 3669, '3': 1809, '4-9': 3357, '10-99': 2113, '100+': 173}
irregular df histogram: {'1': 11668, '2': 0, '3': 0, '4-9': 0, '10-99': 0, '100+': 0}


,term,idf,df,example
0,أعینهم,9.5998,1,وهذا أمر ینبغی أن یضعه أحباء الله نصب أعینهم ح...
1,ارتقائهم,9.5998,1,ولو انّی اعلم بأنّک ارحم بهم من انفسهم و ما اب...
2,اعمل,9.5998,1,ثمّ اعمل بهم ما ینبغی بجودک و کرمک.
3,الأقطع,9.5998,1,فاعرف بالیقین الأقطع و الأمر المثبت الأحتم بأن...
4,الجسدیة,9.5998,1,ولذلک فأهمیّة هذه الفترة وغایتها أساساً روحانی...
5,الحباب,9.5998,1,امشی مقبلاً الی العزیز الوهّاب و ورائی تنساب ا...
6,الخناس,9.5998,1,ثمّ اعصمنا یا محبوب الابداع و مقصود الاختراع ب...
7,الذاکرین,9.5998,1,لک الحمد یا الهی و اله العالمین و مقصودی و مقص...
8,المهداة,9.5998,1,باستثناء الملابس المستعملة للزوجة، والمجوهرات ...
9,المواقع,9.5998,1,لا یجوز ترک الصلاة إلا فی المواقع غیر المأمونة.


### Normalization check — ZWNJ

In [13]:
collisions = json.load(open('results/rarity_train.json'))['zwnj_collisions']
print(f'{len(collisions)} ZWNJ variant collisions')
pd.DataFrame(collisions, columns=['split spelling', 'joined spelling']).head(25)

25 ZWNJ variant collisions


,split spelling,joined spelling
0,آسود‌گی,آسودگی
1,الذین‌هم,الذینهم
2,انشآء‌الله,انشآءالله
3,این‌قدر,اینقدر
4,بیچار‌گان,بیچارگان
5,بی‌خبر,بیخبر
6,بی‌مثال,بیمثال
7,جان‌فزا,جانفزا
8,خون‌ریزی,خونریزی
9,راست‌گو,راستگو


---
## Step 3 — leakage audit

In [14]:
!python3 manage.py leakage --config configs/sparse_retrieval.yaml --split val test --write-quarantine

Auditing 1323 val rows against 10860 pool rows ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 3064.99it/s]
  17/1323 eval rows flagged, 22 pool rows implicated -> results/leakage_val.json
Auditing 1322 test rows against 10860 pool rows ...
  21/1322 eval rows flagged, 28 pool rows implicated -> results/leakage_test.json
Quarantined 50 pool rows (0.46%) -> data/splits/pool_quarantine.json


In [15]:
for split in ('val', 'test'):
    leak = json.load(open(f'results/leakage_{split}.json'))
    print(f"{split}: {leak['n_eval_rows_flagged']}/{leak['n_eval_rows']} eval rows flagged, "
          f"{leak['n_pool_rows_flagged']} pool rows implicated")
    print('  max-cos histogram:', leak['max_cos_histogram'])

worst = json.load(open('results/leakage_val.json'))['flags'][:10]
pd.DataFrame([
    {'cos': f['cos'], 'jac_src': f['jaccard_source'], 'jac_tgt': f['jaccard_target'],
     'eval': f['eval_source'][:50], 'pool': f['pool_source'][:50]}
    for f in worst
])

val: 17/1323 eval rows flagged, 22 pool rows implicated
  max-cos histogram: {'0.00-0.50': 0, '0.50-0.70': 0, '0.70-0.80': 0, '0.80-0.85': 0, '0.85-0.90': 506, '0.90-0.95': 816, '0.95-0.97': 1, '0.97-0.99': 0, '0.99-1.01': 0}
test: 21/1322 eval rows flagged, 28 pool rows implicated
  max-cos histogram: {'0.00-0.50': 0, '0.50-0.70': 0, '0.70-0.80': 0, '0.80-0.85': 1, '0.85-0.90': 663, '0.90-0.95': 653, '0.95-0.97': 5, '0.97-0.99': 0, '0.99-1.01': 0}


,cos,jac_src,jac_tgt,eval,pool
0,0.9508,0.7195,0.2353,امید هست در ظلّ سدرهٴ عنایت الهی تربیت شوید و ...,امید هست در ظلّ سدرهٔ عنایت الهیّه تربیت شوید ...
1,0.9451,0.8319,0.8779,چون که هر روز را امری و هر حین را حکمی مقتضی ل...,چون که هر روز را امری و هر حین را حکمتی مقتضی ...
2,0.9394,0.7059,0.6381,تمسّکوا بحبل الأسباب متوکّلین علی الله مسبّب ا...,تمسّکوا بحبل الأسباب متوکلین على الله مسبّب ال...
3,0.9331,0.7209,0.4286,انّک انت القویّ المقتدر العزیز المتین.,انّک انت المقتدر المتعالی القویّ العزیز العظیم.
4,0.9301,0.7313,0.8491,یا حزب الله مربّی عالم عدل است چه که دارای دو ...,مربّی عالم عدلست چه که دارای دو رکن است مجازات...
5,0.9298,0.5897,0.7500,ابغض النّاس عند الله من یقعد و یطلب.,أبغض الناس عند الله من یقعد ویطلب.
6,0.9267,0.9459,0.4831,انّک انت المقتدر العزیز المهیمن القیّوم.,و انّک انت المقتدر المهیمن العزیز القیّوم.
7,0.9251,0.8611,0.3600,و لا یسأل عمّا یفعل و کلّ عن کلّ یسألون.,انّه لا یسأل عمّا یفعل و کلّ عن کلّ یسألون.
8,0.9209,0.7308,0.7674,انّ ربّک لهو العلیم الحکیم.,انّ ربّک هو العلیم الحکیم.
9,0.9205,0.9048,0.5909,و الحمد لله ربّ العالمین.,الحمد لله ربّ العالمین!


In [16]:
!python3 manage.py build_index --config configs/base_qwen.yaml \
    --index_dir data/knn_index_clean \
    --quarantine data/splits/pool_quarantine.json

Quarantine data/splits/pool_quarantine.json: dropped 50 of 10860 rows
Embedding 10810 Persian/Arabic training sources with intfloat/multilingual-e5-large-instruct ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Batches: 100%|███████████████████████████████| 338/338 [00:13<00:00, 25.08it/s]
Wrote index to data/knn_index_clean/ : embeddings (10810, 1024), 10810 pairs


---
## Step 2 — the sparse channel

In [17]:
!python3 manage.py sparse_select --config configs/sparse_retrieval.yaml \
    --split val --index_dir data/knn_index_clean

Selecting k=8 for 1323 val sources over data/knn_index_clean ...
Loading intfloat/multilingual-e5-large-instruct on device: cuda
Loading weights: 100%|█████████████████████| 391/391 [00:00<00:00, 3021.68it/s]
Dense-only baseline for the redundancy comparison ...
  routes: {'sparse': 0.0, 'hybrid': 0.0197, 'dense': 0.9803}
  queries with >= n irregular terms: {'1': 0.4981, '2': 0.1882, '3': 0.0582, '4': 0.0197, '5': 0.0091, '6': 0.003}
  mean coverage (routed): 1.0
  intra-set cosine: {'sparse': 0.8896, 'dense_baseline': 0.9002}
Wrote results/sparse_selection_val.json


In [18]:
sel = json.load(open('results/sparse_selection_val.json'))
print('routes            :', sel['route_fractions'])
print('irregular terms/query — mean', sel['query_terms']['mean'],
      'deciles', sel['query_terms']['deciles'])
print('share at or above :', sel['query_terms']['share_at_or_above'])
print('coverage (routed) :', sel['coverage']['mean'])
print('intra-set cosine  :', sel['intra_set_similarity'])

routes            : {'sparse': 0.0, 'hybrid': 0.0197, 'dense': 0.9803}
irregular terms/query — mean 0.777 deciles [0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 7]
share at or above : {'1': 0.4981, '2': 0.1882, '3': 0.0582, '4': 0.0197, '5': 0.0091, '6': 0.003}
coverage (routed) : 1.0
intra-set cosine  : {'sparse': 0.8896, 'dense_baseline': 0.9002}


### Threshold sweep

`min_query_terms` is the one knob that decides whether the channel exists at all. Selection
is cheap once the index is loaded, so sweep it rather than arguing about it. Each run
writes its own report, leaving the configured run's above intact.

In [19]:
for thr in (1, 2, 3, 4):
    print(f'--- min_query_terms={thr}')
    !python3 manage.py sparse_select --config configs/sparse_retrieval.yaml --split val --index_dir data/knn_index_clean --min_query_terms {thr} --out results/sparse_sweep_val_t{thr}.json 2>&1 | grep -E 'routes|intra-set'

--- min_query_terms=1
  routes: {'sparse': 0.0, 'hybrid': 0.4958, 'dense': 0.5042}
  intra-set cosine: {'sparse': 0.8849, 'dense_baseline': 0.9002}
--- min_query_terms=2
  routes: {'sparse': 0.0, 'hybrid': 0.1875, 'dense': 0.8125}
  intra-set cosine: {'sparse': 0.8871, 'dense_baseline': 0.9002}
--- min_query_terms=3
  routes: {'sparse': 0.0, 'hybrid': 0.0582, 'dense': 0.9418}
  intra-set cosine: {'sparse': 0.8888, 'dense_baseline': 0.9002}
--- min_query_terms=4
  routes: {'sparse': 0.0, 'hybrid': 0.0197, 'dense': 0.9803}
  intra-set cosine: {'sparse': 0.8896, 'dense_baseline': 0.9002}


### Worked examples

The trace behind three routed queries: which irregular terms the query carried, how much
of that rarity the selected set covered, and which exemplars were chosen.

In [20]:
for ex in sel['examples']:
    print('QUERY :', ex['source'][:90])
    print('  route', ex['trace']['route'], '| terms', ex['trace']['query_terms'],
          '| coverage', ex['trace']['coverage'])
    for e in ex['exemplars']:
        print('   -', e[:90])
    print()

QUERY : لذا اذکر لک بعض ما اکرمنی الله عمّا تطیقه النّفوس و تحمله العقول لئلّا یرفع ضوضآء المبغضین
  route hybrid | terms ['المبغضین', 'المنافقین', 'اکرمنی', 'تحمله'] | coverage 1.0
   - فارحمنی بجودک ثمّ اکرمنی بسلطانک ثمّ قرّبنی بألطافک.
   - ولکن انّا لا نحبّ بأن نذکر ما لا ذکر فی البیان لئلّا یرفع ضجیج المبغضین.
   - ان اثبتنی علی حبّک و رضائک علی شأن لا یمنعنی اعراض المشرکین من بریّتک و ضوضآء المنافقین من
   - لو ارید ان اذکر لک ما ورد علیّ لن تحمله النّفوس و لا العقول
   - یا حزب الله جهد نمائید شاید قلوب احزاب مختلفهٔ عالم بآب بردباری و شفقت شما از ضغینه و بغضا
   - کذلک نطق لسانی لأحد اغصانی و ذکرناه لأحبّائی الّذین نبذوا الأوهام و اخذوا ما امروا به فی ی
   - و امّا من چنین میگویم دشمنانتان را دوست دارید و ذکر خیر کنید بدگویان خود را و مبغضانتان را
   - ویلٌ لک یا ایُّها المشرک باللّه و للّذین اتّخذوک إماما لأنفسهم من دون بیّنة و لا کتاب مشهو

QUERY : و ان یقولون هذه الأسفار الّتی تکون بین یدی هذه الفئة و یسمّونها بالانجیل و ینسبونها بعیسی 
  route hybrid | terms ['الفئة', 'الف